## 3. Collapse thresholds, finite-time extinction and basins
<div style="border-left:4px solid #16803d;padding:10px"><strong>WP03 analysis implemented.</strong> Synthetic deterministic models; final group integration, human reviews and Devam's detailed learning review remain pending.</div>

For a well-mixed population, assume constant $r>0$, $0<A<K$, nonnegative harvest, no delays/age structure or other species. Put $x=N/K,\ \tau=rt,\ a=A/K,\ q=H/(rK),\ e=h/r$. Then
$$
x'=G(x)-q\quad(A),\qquad x'=G(x)-ex\quad(B),\qquad
G(x)=-x^3/a+(1+a)x^2/a-x.
$$
**Threshold derivation.** $G'=(-3x^2+2(1+a)x-a)/a$. The larger stationary point is
$x_*=(1+a+\sqrt{1-a+a^2})/3$, in $(a,1)$, with $G''(x_*)<0$;
the smaller point lies below a. Thus $q_c=G(x_*)$ and
$N_*=(K+A+\sqrt{K^2-KA+A^2})/3,\ H_c=rKq_c$.
For B, maximise $P(x)=G(x)/x=(1-x)(x/a-1)$:
$P'=(1+a-2x)/a,\ P''=-2/a$, so
$e_c=(1-a)^2/(4a)$, $h_c=re_c$.
The logistic class maximum is $1/4$, giving $H_{c,\mathrm{logistic}}=rK/4$.
H and h have different units; threshold comparisons also depend on production calibration.

**Algorithm:** compute closed-form thresholds; check them against independent bounded numerical maxima of expanded polynomials; classify basins before any extinction integration.

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))

### Basins and critical cases
For $0<q<q_c$, A has two positive roots $x_-<x_+$ and signs $-,+,-$: below the lower root extinction, above it approach to the upper root. Exact roots remain equilibria. For B,
$$
x_\pm=(1+a\pm\sqrt{(1-a)^2-4ae})/2.
$$
At positive roots $f'_B=xP'$, so the lower is unstable and upper stable; zero has derivative $-(1+e)<0$.

| Control | A | B |
|---|---|---|
| Zero | Below a approaches zero asymptotically; above a tends to 1 | Same positive-state basins |
| Below fold | Below lower root reaches zero; above tends to upper root | Below lower root approaches zero; above tends to upper root |
| At fold | Below double root reaches zero; above approaches it | Below double root approaches zero; above approaches it |
| Above fold | Every finite positive start reaches zero | Every positive start approaches zero |

Exact initial equilibria stay there in all rows. At a fold the derivative is zero but signs on both sides are negative: **semistable**, attracting only from above.
The classifier explicitly recognises returned roots and marks non-exact starts within $2\,10^{-12}$ of an unstable/semistable boundary unresolved. It does not merge adjacent harvest controls into the fold or decide survival from a finite horizon.

Numerically, compute the B upper root first and use the quadratic root product `lower = a*(1+e)/upper` to avoid cancellation for small a; the exact fold remains an explicit repeated root. The integrated WP01–WP03 APIs use the same threshold and branch providers and exact equality to the computed critical control. Adjacent controls retain distinct topology; numerical and initial-state uncertainty are reported separately.


### Extinction integral and solver checks
On a reaching-zero A path,
$$
\tau_{\rm ext}=\int_0^{x_0}\frac{du}{q-G(u)},\qquad
t_{\rm ext}=\tau_{\rm ext}/r=\int_0^{N_0}\frac{dN}{H-g(N)}.
$$
For a subcritical start below the barrier the denominator is continuous and positive over the whole path; at/above an intervening equilibrium the extinction integral is inapplicable. For $q>q_c$, $G\le q_c$ gives $\tau_{\rm ext}\le x_0/(q-q_c)$.
For zero-yield A and B, smoothness/uniqueness exclude a positive trajectory reaching zero in finite time.

**Algorithm:** classify -> integrate only reaching-zero paths -> compare quadrature with two RK45 tolerances and independently expanded-polynomial DOP853 -> retain every case/status. No equilibrium pole is crossed. Blank times mean no finite extinction or unresolved boundary, not zero.

RK45 uses a shared-stage 5(4) pair for adaptive error control. We take the fifth-order state, locate the first downward zero event for positive-yield A, and include its endpoint without adding a zero tail. Output times are separate from adaptive internal steps. Quadrature error estimates and local RK tolerances are not rigorous global error bounds. References: [SciPy solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [quad](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html).

**Difficult endpoint handling:** near the subcritical barrier, a logarithmic variable removes the vanishing linear factor. At the computed fold an elementary partial-fraction expression evaluates the extinction integral (`fold_closed_form`, no quadrature-error estimate). This avoids missed endpoint contributions for close-below-fold starts. Root rounding is separate from integration error. See `docs/WP03_EXPLAINED.md` for a basics-first explanation and the derivation.

Near a subcritical boundary, $d\tau_{\rm ext}/dx_0 = 1/[q-G(x_0)] \sim 1/[f_A'(x_-)(x_--x_0)]$. Rounding in the initial state or root is amplified as the gap shrinks; small integration error alone does not justify extra digits.


In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json"),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json"),
]
for label, script, configuration in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))

### Interpret the synthetic curves
The default $r=0.7,K=100,A=20$ gives $N_*=70.55050$, $H_c=36.75966$, $h_c=0.56$; logistic gives $17.5$.
The 427 extinction evaluations retain non-finite outcomes rather than turning them into zero. Initial zero is omitted on the logarithmic time axis. Solid basin boundaries mark critical initial populations; the finite colour grid does not replace exact equilibrium classification.

At fixed reaching-zero initial population, increasing catch reduces extinction time. Approaching a basin boundary can make that time very large. A finite simulation horizon without an event does not settle whether extinction happens later. The six event checks include the fold from below and supercritical passage; the observed differences are much smaller than the declared $2\,10^{-7}+5\,10^{-7}\tau_{\rm ext}$ acceptance tolerance.

Basin colour regions meet exactly at control/critical = 1; the fold is drawn as a line, so persistence cells do not extend above it. Every WP03 figure is exported as PNG/EPS/vector PDF; extensionless report references select the PDF under pdflatex.


### Why collapse can look delayed: the bottleneck
With $q=q_c+\delta,\ y=x-x_*$, the exact cubic expansion is
$$
q-G(x)=\delta+\beta y^2+y^3/a,\qquad \beta=\sqrt{1-a+a^2}/a.
$$
For a fixed start above $x_*$, the trajectory crosses a slow region. Integrating the leading denominator gives
$$
\int\frac{dy}{\delta+\beta y^2}
=\frac{\arctan(y\sqrt{\beta/\delta})}{\sqrt{\beta\delta}},
\quad \tau_{\rm ext}\sim\pi/\sqrt{\beta\delta}.
$$
The full near-fold integral is evaluated with $y=\sqrt{\delta/\beta}\tan\theta$ to resolve its narrow peak.
This is passage after the equilibria disappear, distinct from relaxation to a stable equilibrium. A start already below the bottleneck need not have this divergence. At the fold, approach from above is algebraic, $y\simeq y_0/(1+\beta y_0\tau)$.

In [ ]:
rows = read_rows("wp03_bottleneck.csv")
for row in rows:
    print("delta/qc =", row["fraction_above_fold"], "| tau*sqrt(delta) =", row["scaled_time"])
print("Leading scaled-time limit:", rows[0]["asymptotic_scaled_time"])
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))

### Optional results: sensitivity and sustainable yield
Holding other dimensional r,K,A fixed, $g_N(N_*)=0$ eliminates the derivative of $N_*$ from the maximum:
$$
H_{c,r}=H_c/r,\quad H_{c,K}=rN_*^2(N_*/A-1)/K^2,\quad
H_{c,A}=-rN_*^2(1-N_*/K)/A^2.
$$
Also
$$
h_{c,r}=h_c/r,\quad h_{c,K}=\tfrac r4(1/A-A/K^2),\quad
h_{c,A}=\tfrac r4(-K/A^2+1/K).
$$
Central differences check these derivatives; elasticities are $(p/C)C_p$. An a sweep at fixed r,K is a different experiment.

A's stable-branch yield supremum $H_c$ lies at the semistable cliff, with no control margin.
For B, $Y(h)=hN_+(h)=g(N_+(h))$. The production maximiser is on the upper branch, so
$h_{\rm MSY}=g(N_*)/N_*<h_c,\ Y_{\rm MSY}=H_c$.
Independent bounded numerical maximisation checks the optimum. The example effort margin at MSY is about 6.96%; the initial state must still exceed the unstable root. Separate illustrative A/B policies are not a universal risk ranking.

In [ ]:
summary = json.loads((outputs["analysis"] / "wp03_summary.json").read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(outputs["analysis"] / "wp03_sensitivity_yield.png")))
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))

### Handoff, reproduction and limitations
The early handoff retains six clean A/B CSVs (606 points). WP05 owns noise, frozen splits, inverse PINN/ODE fitting and held-out validation; WP04 supplies the shared neural engine. Keep known conditions/fixed external scales separate from evaluation truth, and never fit against held-out clean values or hidden unknown parameters.

Exact commands and test/evidence mapping: **docs/WP03_ANALYSIS.md**. The experiment regenerates tables and PNG/EPS; manifests record versions, units, tolerances and source/file hashes. All data are synthetic; no empirical species-level conclusion follows.

The report, notebook and EPS files required by the course must ultimately be integrated with all five tasks. **scripts/check_submission_ready.py** aggregates missing final artifacts, unfinished sections and metadata; a successful WP03 run does not make the final group submission ready. Reviews and non-author reproduction remain required.

AI assistance: OpenAI ChatGPT/Codex for this implementation, mathematical writing, tests and documentation, recorded in docs/AI_USAGE.md. Devam's detailed understanding review remains pending as requested.